In [3]:
# ============================================
# Stock Market Data Analysis
# Author: Samaneh Kavianfar
# Project: Stock Analysis with OOP
# ============================================

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

In [ ]:

# ============================================
# Class 1: DataLoader
# ============================================
class DataLoader:
    """
    Responsible for downloading and cleaning stock data.
    """
    
    def __init__(self, symbols, start_date, end_date):
        self.symbols = symbols
        self.start_date = start_date
        self.end_date = end_date
        self.data = None
    
    def download_data(self):
        """Download stock data from Yahoo Finance."""
        print("\n📥 Downloading data...")
        all_data = []
        
        for symbol in self.symbols:
            print(f" ⬇️ Downloading {symbol}...")
            ticker = yf.Ticker(symbol)
            df = ticker.history(start=self.start_date, end=self.end_date)
            
            if df.empty:
                print(f" ⚠️ No data found for {symbol}")
                continue
                
            df = df.reset_index()
            df['Symbol'] = symbol
            df = df[['Date', 'Symbol', 'Open', 'High', 'Low', 'Close', 'Volume']]
            all_data.append(df)
            print(f" ✅ {len(df)} rows downloaded for {symbol}")
        
        if not all_data:
            raise ValueError("❌ No data downloaded for any symbol")
        
        self.data = pd.concat(all_data, ignore_index=True)
        print(f"\n✅ Total data: {len(self.data)} rows")
        print(f"📊 Companies: {self.data['Symbol'].nunique()}")
        return self.data
    
    def clean_data(self):
        """Clean the data: missing values and duplicates."""
        print("\n🧹 Cleaning data...")
        
        if self.data is None:
            raise ValueError("❌ Please run download_data() first")
        
        # Check for missing values
        missing = self.data.isnull().sum()
        if missing.sum() > 0:
            print(f" ⚠️ Found missing values: {missing[missing > 0].to_dict()}")
            self.data = self.data.dropna()
            print(f" ✅ Dropped rows with missing values")
        else:
            print(f" ✅ No missing values found")
        
        # Check for duplicates
        duplicates = self.data.duplicated().sum()
        if duplicates > 0:
            print(f" ⚠️ Found {duplicates} duplicate rows")
            self.data = self.data.drop_duplicates()
            print(f" ✅ Dropped duplicates")
        else:
            print(f" ✅ No duplicates found")
        
        # Convert Date to datetime
        self.data['Date'] = pd.to_datetime(self.data['Date'])
        
        print(f"✅ Cleaned data: {len(self.data)} rows")
        return self.data

In [ ]:
# ============================================
# Test DataLoader Class
# ============================================

# Define parameters
symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
start_date = '2024-01-01'
end_date = '2024-12-31'

# Create an instance of DataLoader
loader = DataLoader(symbols, start_date, end_date)

# Download data
print("📥 Step 1: Downloading data...")
raw_data = loader.download_data()

# Check the shape of the data
print(f"\n📊 Shape of raw data: {raw_data.shape}")
print(f"📊 Columns: {raw_data.columns.tolist()}")

# Show first 5 rows
print("\n📋 First 5 rows of raw data:")
print(raw_data.head())

# Clean the data
print("\n🧹 Step 2: Cleaning data...")
clean_data = loader.clean_data()

# Check the shape after cleaning
print(f"\n📊 Shape after cleaning: {clean_data.shape}")

# Check for missing values
print(f"\n📊 Missing values after cleaning:")
print(clean_data.isnull().sum())

# Show first 5 rows after cleaning
print("\n📋 First 5 rows after cleaning:")
print(clean_data.head())

In [ ]:

# ============================================
# Class 2: StockAnalyzer
# ============================================
class StockAnalyzer:
    """
    Responsible for calculating returns, metrics, and correlations.
    """
    
    def __init__(self, data):
        self.data = data
        self.returns = None
        self.metrics = None
        self.correlation = None
    
    def calculate_returns(self):
        """Calculate daily returns for each stock."""
        print("\n📈 Calculating daily returns...")
        
        if self.data is None:
            raise ValueError("❌ Please load data first")
        
        pivot_data = self.data.pivot(index='Date', columns='Symbol', values='Close')
        daily_returns = pivot_data.pct_change().dropna()
        
        self.returns = daily_returns.reset_index().melt(
            id_vars=['Date'], 
            var_name='Symbol', 
            value_name='Daily_Return'
        ).dropna()
        
        print(f"✅ Daily returns calculated for {self.returns['Symbol'].nunique()} symbols")
        return self.returns
    
    def calculate_metrics(self):
        """Calculate key metrics: total return, volatility, Sharpe ratio."""
        print("\n📊 Calculating key metrics...")
        
        if self.returns is None:
            raise ValueError("❌ Please run calculate_returns() first")
        
        metrics_list = []
        
        for symbol in self.returns['Symbol'].unique():
            symbol_returns = self.returns[self.returns['Symbol'] == symbol]['Daily_Return']
            
            if len(symbol_returns) < 2:
                continue
                
            total_return = (1 + symbol_returns).prod() - 1
            annualized_return = (1 + total_return) ** (252 / len(symbol_returns)) - 1
            volatility = symbol_returns.std() * np.sqrt(252)
            sharpe_ratio = annualized_return / volatility if volatility > 0 else 0
            
            metrics_list.append({
                'Symbol': symbol,
                'Total_Return': total_return,
                'Annualized_Return': annualized_return,
                'Volatility': volatility,
                'Sharpe_Ratio': sharpe_ratio,
                'Days': len(symbol_returns)
            })
        
        self.metrics = pd.DataFrame(metrics_list)
        print(f"✅ Metrics calculated for {len(self.metrics)} symbols")
        return self.metrics
    
    def calculate_correlation(self):
        """Calculate correlation between all stock returns."""
        print("\n🔗 Calculating correlation matrix...")
        
        if self.returns is None:
            raise ValueError("❌ Please run calculate_returns() first")
        
        pivot_returns = self.returns.pivot(index='Date', columns='Symbol', values='Daily_Return')
        self.correlation = pivot_returns.corr()
        
        print("✅ Correlation matrix calculated")
        return self.correlation

In [ ]:
# ============================================
# Test StockAnalyzer Class
# ============================================

# Create an instance of StockAnalyzer with the cleaned data
analyzer = StockAnalyzer(clean_data) # clean_data comes from the previous cell

# Calculate returns
returns = analyzer.calculate_returns()

# Show first 5 rows of returns
print("\n📋 First 5 rows of daily returns:")
print(returns.head())

# Calculate metrics
metrics = analyzer.calculate_metrics()

# Show metrics
print("\n📊 Key Metrics:")
print(metrics)

# Calculate correlation
correlation = analyzer.calculate_correlation()

# Show correlation matrix
print("\n🔗 Correlation Matrix:")
print(correlation)

In [2]:

# ============================================
# Class 3: ReportGenerator
# ============================================
class ReportGenerator:
    """
    Responsible for saving and displaying results.
    """
    
    def __init__(self, data, returns, metrics, correlation):
        self.data = data
        self.returns = returns
        self.metrics = metrics
        self.correlation = correlation
    
    def save_results(self):
        """Save all results to CSV files."""
        print("\n💾 Saving results...")
        
        self.data.to_csv('stock_data_cleaned.csv', index=False)
        print(f" ✅ Saved: stock_data_cleaned.csv")
        
        self.returns.to_csv('stock_daily_returns.csv', index=False)
        print(f" ✅ Saved: stock_daily_returns.csv")
        
        self.metrics.to_csv('stock_metrics.csv', index=False)
        print(f" ✅ Saved: stock_metrics.csv")
        
        if self.correlation is not None:
            self.correlation.to_csv('stock_correlation.csv')
            print(f" ✅ Saved: stock_correlation.csv")
        
        print("\n✅ All results saved successfully!")
    
    def display_summary(self):
        """Display a summary of the results."""
        print("\n" + "="*60)
        print("📋 SUMMARY OF KEY METRICS")
        print("="*60)
        
        if self.metrics is not None:
            print(self.metrics.to_string(index=False))
        else:
            print("⚠️ No metrics available. Run calculate_metrics() first.")
        
        if self.correlation is not None:
            print("\n" + "="*60)
            print("🔗 CORRELATION MATRIX")
            print("="*60)
            print(self.correlation)



📥 Downloading data...
 ⬇️ Downloading AAPL...
 ✅ 751 rows downloaded for AAPL
 ⬇️ Downloading MSFT...
 ✅ 751 rows downloaded for MSFT
 ⬇️ Downloading GOOGL...
 ✅ 751 rows downloaded for GOOGL
 ⬇️ Downloading AMZN...
 ✅ 751 rows downloaded for AMZN
 ⬇️ Downloading TSLA...
 ✅ 751 rows downloaded for TSLA
 ⬇️ Downloading NVDA...
 ✅ 751 rows downloaded for NVDA
 ⬇️ Downloading JPM...
 ✅ 751 rows downloaded for JPM
 ⬇️ Downloading NKE...
 ✅ 751 rows downloaded for NKE

✅ Total data: 6008 rows
📊 Companies: 8

🧹 Cleaning data...
 ✅ No missing values found
 ✅ No duplicates found
✅ Cleaned data: 6008 rows

📈 Calculating daily returns...
✅ Daily returns calculated for 8 symbols

📊 Calculating key metrics...
✅ Metrics calculated for 8 symbols

🔗 Calculating correlation matrix...
✅ Correlation matrix calculated

💾 Saving results...
 ✅ Saved: stock_data_cleaned.csv
 ✅ Saved: stock_daily_returns.csv
 ✅ Saved: stock_metrics.csv
 ✅ Saved: stock_correlation.csv

✅ All results saved successfully!

📋 SU

In [ ]:
# ============================================
# Test ReportGenerator Class
# ============================================

# Create an instance of ReportGenerator
report = ReportGenerator(clean_data, returns, metrics, correlation)

# Save all results
report.save_results()

# Display summary
report.display_summary()

# Check if files are created
import os
print("\n📂 Files created:")
files = ['stock_data_cleaned.csv', 'stock_daily_returns.csv', 'stock_metrics.csv', 'stock_correlation.csv']
for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f" ✅ {file} (size: {size} bytes)")
    else:
        print(f" ❌ {file} not found")

In [ ]:
# ============================================
# Run the complete pipeline
# ============================================

def run_stock_analysis(symbols, start_date, end_date):
    """
    Run the complete stock analysis pipeline.
    """
    print("🚀 Starting Stock Analysis Pipeline...")
    print("="*50)
    
    # Step 1: Load and clean data
    loader = DataLoader(symbols, start_date, end_date)
    raw_data = loader.download_data()
    clean_data = loader.clean_data()
    
    # Step 2: Analyze data
    analyzer = StockAnalyzer(clean_data)
    returns = analyzer.calculate_returns()
    metrics = analyzer.calculate_metrics()
    correlation = analyzer.calculate_correlation()
    
    # Step 3: Generate report
    report = ReportGenerator(clean_data, returns, metrics, correlation)
    report.save_results()
    report.display_summary()
    
    print("\n✅ Pipeline completed successfully!")

# Run the pipeline
run_stock_analysis(
    symbols=['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA'],
    start_date='2024-01-01',
    end_date='2024-12-31'
)

In [ ]:

# ============================================
# Main Execution
# ============================================
if __name__ == "__main__":
    # Define parameters
    symbols = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'JPM', 'NKE']
    start_date = '2023-01-01'
    end_date = '2025-12-31'
    
    # Step 1: Load and clean data
    loader = DataLoader(symbols, start_date, end_date)
    raw_data = loader.download_data()
    clean_data = loader.clean_data()
    
    # Step 2: Analyze data
    analyzer = StockAnalyzer(clean_data)
    returns = analyzer.calculate_returns()
    metrics = analyzer.calculate_metrics()
    correlation = analyzer.calculate_correlation()
    
    # Step 3: Generate report
    report = ReportGenerator(clean_data, returns, metrics, correlation)
    report.save_results()
    report.display_summary()